# Live Yahoo lineup workflow

This notebook uses the project modules directly: load config, fetch your Yahoo roster, enrich it with MLB starting-status checks, and then run the optimizer. By default it stays in dry-run mode; a later cell lets you opt into a live Yahoo write and refetch the roster.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import load_config
from lineup import (
    optimize_lineup,
    render_plan,
    render_roster,
    is_bench_position,
    player_should_be_replaced,
    choose_replacement,
    can_fill_position,
    apply_plan_to_roster
)
from mlb_lineups import clear_caches, enrich_roster_with_starting_status
from yahoo_api import YahooFantasyClient, build_roster_update_xml

In [2]:
TARGET_DATE = '2026-03-26'
VERBOSE = False
IGNORE_LOCKS = False

config = load_config(lineup_date=TARGET_DATE, apply_changes=False)
client = YahooFantasyClient(config)
raw_roster = client.get_team_roster(config.lineup_date)
print(f'Fetched {len(raw_roster.players)} players for {raw_roster.team_name}.')


Fetched 29 players for deGrom Reapers.


In [3]:
clear_caches()
roster = enrich_roster_with_starting_status(raw_roster, date_str=config.lineup_date, verbose=VERBOSE, ignore_locks=IGNORE_LOCKS)
print(render_roster(roster))

Team: deGrom Reapers
Date: 2026-03-26
Slots: C x1, 1B x1, 2B x1, 3B x1, SS x1, IF x1, LF x1, CF x1, RF x1, OF x1, Util x1, SP x3, RP x3, P x3
Roster:
- 1B   Josh Naylor (1B) [lineup pending]
- 2B   Caleb Durbin (2B,3B) [starting]
- 3B   Alec Bohm (1B,3B) [starting]
- C    Shea Langeliers (C) [no game]
- CF   Pete Crow-Armstrong (CF) [starting]
- IF   Zach Neto (SS) [starting]
- LF   Wyatt Langford (LF,CF) [starting]
- OF   Jung Hoo Lee (CF) [no game]
- P    Drew Rasmussen (SP) [starting]
- P    Emmet Sheehan (SP) [not starting]
- P    Kodai Senga (SP) [not starting; locked]
- RF   Tyler O'Neill (LF,RF) [starting]
- RP   Edwin Díaz (RP) [reliever]
- RP   Emilio Pagán (RP) [reliever]
- RP   Matt Svanson (RP) [reliever]
- SP   Hunter Brown (SP) [starting]
- SP   Jacob deGrom (SP) [not starting]
- SP   Joe Ryan (SP) [starting]
- SS   Elly De La Cruz (SS) [starting]
- Util Shohei Ohtani (Batter) (Util) [lineup pending]
- BN   Cole Young (2B) [lineup pending]
- BN   Dustin May (SP) [not star

In [4]:
plan = optimize_lineup(roster)
print(render_plan(plan))

Plan:
- Warning: No eligible starting replacement found for Shea Langeliers at C.
- Warning: No eligible starting replacement found for Jung Hoo Lee at OF.
- Warning: No eligible starting replacement found for Jacob deGrom at SP.
- Warning: No eligible starting replacement found for Emmet Sheehan at P.
- Warning: Kodai Senga is locked at P; no change attempted.
- No changes proposed.


In [5]:
updated_roster = apply_plan_to_roster(roster, plan)
print(render_roster(updated_roster))


Team: deGrom Reapers
Date: 2026-03-26
Slots: C x1, 1B x1, 2B x1, 3B x1, SS x1, IF x1, LF x1, CF x1, RF x1, OF x1, Util x1, SP x3, RP x3, P x3
Roster:
- 1B   Josh Naylor (1B) [lineup pending]
- 2B   Caleb Durbin (2B,3B) [starting]
- 3B   Alec Bohm (1B,3B) [starting]
- C    Shea Langeliers (C) [no game]
- CF   Pete Crow-Armstrong (CF) [starting]
- IF   Zach Neto (SS) [starting]
- LF   Wyatt Langford (LF,CF) [starting]
- OF   Jung Hoo Lee (CF) [no game]
- P    Drew Rasmussen (SP) [starting]
- P    Emmet Sheehan (SP) [not starting]
- P    Kodai Senga (SP) [not starting; locked]
- RF   Tyler O'Neill (LF,RF) [starting]
- RP   Edwin Díaz (RP) [reliever]
- RP   Emilio Pagán (RP) [reliever]
- RP   Matt Svanson (RP) [reliever]
- SP   Hunter Brown (SP) [starting]
- SP   Jacob deGrom (SP) [not starting]
- SP   Joe Ryan (SP) [starting]
- SS   Elly De La Cruz (SS) [starting]
- Util Shohei Ohtani (Batter) (Util) [lineup pending]
- BN   Cole Young (2B) [lineup pending]
- BN   Dustin May (SP) [not star

In [6]:
APPLY_TO_YAHOO = False

if not plan.moves:
    print('No changes to apply.')
else:
    print('Planned moves:')
    for move in plan.moves:
        print(f'- {move.player_name}: {move.from_position} -> {move.to_position}')
    print('\nOutgoing Yahoo roster XML:')
    print(build_roster_update_xml(roster.lineup_date or config.lineup_date, plan.moves))
    if APPLY_TO_YAHOO:
        client.set_lineup(
            lineup_date=roster.lineup_date or config.lineup_date,
            moves=plan.moves,
        )
        print('\nApplied lineup changes to Yahoo. Refetching roster...')
        refreshed_raw_roster = client.get_team_roster(config.lineup_date)
        clear_caches()
        refreshed_roster = enrich_roster_with_starting_status(
            refreshed_raw_roster,
            date_str=config.lineup_date,
            verbose=VERBOSE,
            ignore_locks=IGNORE_LOCKS,
        )
        print(render_roster(refreshed_roster))
    else:
        print('\nAPPLY_TO_YAHOO is False. No live changes sent.')


No changes to apply.
